# Gather gene annotations produced by each AMG prediction tool

Using annotations produced when running CheckAMG, DRAM-V, and VIBRANT in the notebook `amg_benchmark_predictions.ipynb`. Including annotations/hits that didn't end up in each tool's final/filtered outputs.

In [ ]:
import importlib.metadata

print({package: importlib.metadata.version(package) for package in ("polars",)})

In [ ]:
import os
from pathlib import Path

ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks")
MAIN_DIR = ROOT_DIR.joinpath("metabolism_benchmark")

THREADS = 50
TMP_DIR = Path("/storage2/scratch/kosmopoulos/tmp/checkamg_notebooks")
TMP_DIR.mkdir(parents=True, exist_ok=True)
AMG_TABLES_DIR = MAIN_DIR.joinpath("amg_tables")
GENE_ID_MAPPING = MAIN_DIR.joinpath("dramv_genes_reformatted", "gene_id_mapping.parquet")
CHECKAMG_HMM_NAMES = Path("../CheckAMG/files/hmm_id_to_name.tsv")
DRAM_DESCRIPTION_DB = Path("/storage2/databases/DRAM_v1.5.0/description_db.sqlite")


## Load gene predictions

Construct polars LazyFrames but do not collect right away because of their size. `gene_id_mapping.parquet` is restricted on load to the 21 benchmark samples, those of the subsets `complete_virus_genomes`, `metagenomes` and `viromes`, each with VirSorter2 outputs.

In [ ]:
os.environ["POLARS_MAX_THREADS"] = str(THREADS)
os.environ["POLARS_TEMP_DIR"] = str(TMP_DIR)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

In [ ]:
import sys

sys.path.insert(0, "./accessory_scripts")
from sample_types import assert_benchmark_samples, benchmark_gene_id_mapping

gene_id_mapping = benchmark_gene_id_mapping(GENE_ID_MAPPING)
BENCHMARK_SAMPLES = gene_id_mapping["sample"].unique().to_list()

In [ ]:
def reformat_for_merge(df: pl.DataFrame) -> pl.DataFrame:
    return df.drop(["subset", "old_gene", "old_scaffold", "metadata"]).rename({
        "new_gene": "gene",
        "new_scaffold": "scaffold",
    })

### From CheckAMG annotate

Requires CheckAMG annotate to have been run on each sample with the flag `--keep-full-hmm-results`.

In [ ]:
CHECKAMG_DIR = MAIN_DIR.joinpath("checkamg_annotate_v1.1_outputs")

In [ ]:
pattern_checkamg_all_annots = "*/*/wdir/hmm_results.parquet"
paths_checkamg_all_annots = sorted(CHECKAMG_DIR.glob(pattern_checkamg_all_annots))

dfs_checkamg_all_annots = []

for p in paths_checkamg_all_annots:
    parts = p.parts
    try:
        i = parts.index(CHECKAMG_DIR.name)
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
    except (ValueError, IndexError):
        continue

    df = pl.scan_parquet(p)

    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
    ])

    # Add sample prefixes to gene names since some scaffold names are identical across samples
    df = df.with_columns(
        pl.col("sequence").alias("gene_orig"),
        pl.concat_str([pl.col("sample"), pl.lit("__"), pl.col("sequence")]).alias("sequence"),
    )

    df = df.join(
        reformat_for_merge(gene_id_mapping).lazy(),
        left_on=["sequence", "sample"],
        right_on=["gene", "sample"],
        how="left"
    ).rename({"sequence": "gene"})

    dfs_checkamg_all_annots.append(df)

In [ ]:
checkamg_hmm_id_to_name = pl.scan_csv(
    CHECKAMG_HMM_NAMES,
    separator="\t"
)

In [ ]:
checkamg_all_annots = (
    pl.concat(dfs_checkamg_all_annots, how="diagonal_relaxed")
    .join(checkamg_hmm_id_to_name, left_on="hmm_id", right_on="id", how="left")
    .sort(
        ["source", "sample", "ecosystem", "scaffold", "start", "gene_number"],
        descending=[False, False, False, False, False, False]
    )
    .rename({"db": "database"})
    .select(
        ["source", "sample", "ecosystem", "gene", "database", "hmm_id", "name", "evalue", "score", "alignment_type", "coverage_sequence", "coverage_hmm", "keep", "note"]
    )
    .with_columns([
        pl.col("gene").cast(pl.Utf8),
        pl.col("database").cast(pl.Utf8),
        pl.col("hmm_id").cast(pl.Utf8),
        pl.col("evalue").cast(pl.Float64),
        pl.col("score").cast(pl.Float64),
    ])
)

### From DRAM-V

**This took ~30 mins and used up to 300 GB of RAM**.

Also assumes that the`.b6` files of annotations generated from running `DRAM-v.py annotate` with the flag `--keep_tmp_dir` were moved and organized into their own folder `dramv_annotations`, following the same `<SEQUENCE TYPE>/<SAMPLE>` directory structure as `dramv_outputs`. The following files are needed from each DRAM-V run:

* `cazy_results.unprocessed.b6`
* `gene_peptidase_hits.b6`
* `gene_viral_hits.b6`
* `kofam_hmm_results.unprocessed.b6`
* `peptidase_gene_hits.b6`
* `pfam_output.b6`
* `viral_gene_hits.b6`
* `vogdb_results.unprocessed.b6`

In [ ]:
import sqlite3

database_description_tables = {
    "dbcan_description": "dbCAN",
    "kegg_description": "KEGG",
    "peptidase_description": "MEROPS",
    "pfam_description": "Pfam",
    "viral_description": "RefSeq Viral",
    "vogdb_description": "VOG",
}

def read_sqlite_table_lazy(db_file, table_name, database_name):
    conn = sqlite3.connect(db_file)

    df = pl.read_database(
        f"SELECT id, description FROM {table_name}",
        conn
    )

    conn.close()

    df = (
        df.rename({"id": "hit_id"})
        .with_columns([
            pl.col("hit_id").cast(pl.Utf8),
            pl.lit(database_name).alias("database")
        ])
    )

    return df.lazy()


db_desc_lazy = []

for table, dbname in database_description_tables.items():
    db_desc_lazy.append(
        read_sqlite_table_lazy(
            DRAM_DESCRIPTION_DB,
            table,
            dbname
        )
    )

db_desc = (
    pl.concat(db_desc_lazy)
    .select(["database", "hit_id", "description"])
    .unique(subset=["database", "hit_id"])
)

In [ ]:
DRAMV_DIR = MAIN_DIR.joinpath("dramv_outputs")
DRAMV_ANNOT_DIR = MAIN_DIR.joinpath("dramv_annotations")

In [ ]:
pattern_dramv_raw = "*/*/*.b6"
paths_dramv_raw = sorted(DRAMV_ANNOT_DIR.glob(pattern_dramv_raw))

gene_map = gene_id_mapping.lazy()

db_renames = {
    "kofam": "KEGG",
    "cazy": "dbCAN",
    "pfam": "Pfam",
    "vogdb": "VOG",
    "peptidase": "MEROPS",
    "viral": "RefSeq Viral",
}

import re
def read_hmmsearch_tbl_dram(path: str):
    # Slower than reading into polars directly but robust to formatting issues
    domtable_cols = [
        "target_name", "target_accession", "tlen", 
        "query_name", "query_accession", "qlen",
        "full_evalue", "full_score", "full_bias",
        "domain_n", "domain_of", "c_evalue", "i_evalue",
        "domain_score", "domain_bias",
        "hmm_from", "hmm_to", "ali_from", "ali_to",
        "env_from", "env_to", "acc", "description"
    ]

    domtbl_dict = {x : [] for x in domtable_cols}

    # A data line split by a stray line break is joined with the next one until it has every column
    rows = []
    buffer = ""

    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            candidate = f"{buffer} {line}" if buffer else line
            parts = re.split(r"\s+", candidate, maxsplit=22)

            if len(parts) >= 22:
                rows.append(candidate)
                buffer = ""
            else:
                buffer = candidate

        if buffer:
            rows.append(buffer)

    for line in rows:
        if line.strip().startswith("#") or line.strip() == "":
            continue
        parts = re.split(r"\s+", line.strip(), maxsplit=22)
        for col, part in zip(domtable_cols, parts):
            if col == "query_name":
                part = part.replace(".hmm", "") # remove .hmm suffix from query names
            domtbl_dict[col].append(part)

    df = pl.DataFrame(domtbl_dict)

    df = (
        df
        .select([
            pl.col("target_name").cast(pl.Utf8).alias("target_name"),
            pl.col("query_name").cast(pl.Utf8).alias("query_name"),
            pl.col("query_accession").cast(pl.Utf8).alias("query_accession"),
            pl.col("full_evalue").alias("evalue"),
            pl.col("full_score").alias("bitscore")
        ])
        .with_columns([
            pl.when(pl.col("query_name") == "-")
            .then(None)
            .otherwise(pl.col("query_name"))
            .alias("query_name"),
            
            pl.when(pl.col("query_accession") == "-")
            .then(None)
            .otherwise(pl.col("query_accession"))
            .alias("query_accession"),

            pl.when(pl.col("evalue") == "-")
            .then(None)
            .otherwise(pl.col("evalue"))
            .cast(pl.Float64)
            .alias("evalue"),

            pl.when(pl.col("bitscore") == "-")
            .then(None)
            .otherwise(pl.col("bitscore"))
            .cast(pl.Float64)
            .alias("bitscore"),
        ])
    )

    return df

def read_mmseqs_b6_tbl(path: str) -> pl.LazyFrame:
    df = pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        comment_prefix="#",
        truncate_ragged_lines=True
    )

    df = (
        df
        .select([
            pl.col("column_1").alias("hit_id"),
            pl.col("column_2").alias("gene"),
            pl.col("column_11").cast(pl.Float64).alias("evalue"),
            pl.col("column_12").cast(pl.Float64).alias("bitscore")
        ])
    )

    return df

# Get "raw" annotations (HMM and MMseqs hits with bitscores and evalues before filtering by DRAM-V)
lazy_dfs_raw = []
for p in paths_dramv_raw:
    try:
        source, sample, file_name = p.relative_to(DRAMV_ANNOT_DIR).parts
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
        db = file_name.split(".")[0]
        if db.startswith("gene_"):
            # skip files with gene queries and db targets, use db queries and gene targets
            continue
        else:
            db = db.split("_")[0]
    except (ValueError, IndexError):
        continue
    
    if db.lower() in ["kofam", "cazy", "vogdb"]:
        print(f"Reading {p.relative_to(DRAMV_ANNOT_DIR)} as HMMER output for database {db}")
        df = read_hmmsearch_tbl_dram(str(p))
        df = df.rename({"target_name": "gene", "query_name": "hit_id"}).drop("query_accession")
        # print(f"df: {df.head()}")
    else:
        print(f"Reading {p.relative_to(DRAMV_ANNOT_DIR)} as MMseqs2 output for database {db}")
        df = read_mmseqs_b6_tbl(str(p))
        if db == "pfam":
            # Pfam MMseqs2 outputs hold gene IDs in query_name and accessions in hit_id, swap them
            df = df.rename({"hit_id": "query_name"})
            df = df.rename({"gene": "hit_id", "query_name": "gene"})
        # print(f"df: {df.head()}")

    df = df.with_columns(pl.col("hit_id").cast(pl.Utf8))

    db = db_renames.get(db.lower(), db)
    
    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
        pl.lit(db).alias("database"),
    ])

    # Join hit IDs to their descriptions from the DRAM description database
    df = df.join(
        db_desc.collect(),
        on=["database", "hit_id"],
        how="left"
    ).rename({"description": "hit_desc"})
    
    df = df.with_columns(
        pl.col("hit_desc").cast(pl.Utf8)
    )

    df = df.join(
        gene_map.collect().select(["sample", "old_gene", "new_gene"]),
        left_on=["gene", "sample"],
        right_on=["old_gene", "sample"],
        how="left"
    ).drop("gene").rename({"new_gene": "gene"})

    # Add sample prefixes to gene names since some scaffold names are identical across samples
    df = df.with_columns(
        pl.concat_str([pl.col("sample"), pl.lit("__"), pl.col("gene")]).alias("gene"),
    )
    df = df.with_columns(
        pl.col("gene").cast(pl.Utf8),
        pl.col("hit_id").cast(pl.Utf8),
        pl.col("database").cast(pl.Utf8),
        pl.col("evalue").cast(pl.Float64),
        pl.col("bitscore").cast(pl.Float64),
        
    )

    lazy_dfs_raw.append(df)

In [ ]:
dramv_all_annots = pl.concat(lazy_dfs_raw, how="vertical")

### From VIBRANT

Will need access to the table `VIBRANT_names.tsv` that comes packaged with VIBRANT (can be downloaded [here](https://raw.githubusercontent.com/AnantharamanLab/VIBRANT/refs/heads/master/files/VIBRANT_names.tsv)).

In [ ]:
VIBRANT_DIR = MAIN_DIR.joinpath("vibrant_outputs")
VIBRANT_HMM_NAMES = Path("/storage2/databases/VIBRANT/files/VIBRANT_names.tsv")

In [ ]:
pattern_vibrant_all_annots = "*/*/VIBRANT_genes_reformatted/VIBRANT_HMM_tables_unformatted_genes_reformatted/genes_reformatted_unformatted_*.hmmtbl"
paths_vibrant_all_annots = sorted(VIBRANT_DIR.glob(pattern_vibrant_all_annots))

dfs_vibrant_all_annots = []

vibrant_names = pl.scan_csv(
    VIBRANT_HMM_NAMES,
    separator="\t",
    new_columns=["hit_id", "hit_desc"]
)

def read_hmmsearch_tbl(path: str) -> pl.LazyFrame:
    records = []
    with open(path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split() # splits on any whitespace run
            if len(parts) < 6:
                continue
            records.append({
                "target_name": parts[0],
                "query_name": parts[2] if parts[2] != "-" else None,
                "query_accession": parts[3] if parts[3] != "-" else None,
                "evalue": float(parts[4]) if parts[4] != "-" else None,
                "bitscore": float(parts[5]) if parts[5] != "-" else None,
            })

    return pl.DataFrame(records, schema={
        "target_name": pl.Utf8,
        "query_name": pl.Utf8,
        "query_accession": pl.Utf8,
        "evalue": pl.Float64,
        "bitscore": pl.Float64,
    }).lazy()

for p in paths_vibrant_all_annots:
    parts = p.parts
    try:
        i = parts.index("vibrant_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
        db = parts[i + 5].split("_")[-1].replace(".hmmtbl", "") # "VIBRANT_HMM_tables_unformatted_genes_reformatted"
    except (ValueError, IndexError):
        continue

    df = read_hmmsearch_tbl(str(p))
    if db.lower() == "pfam":
        df = (
            df
            .rename(
                {
                    "target_name": "gene",
                    "query_accession": "hit_id",
                }
            )
            .drop("query_name")
        )
    else:
        df = (
            df
            .rename(
                {
                    "target_name": "gene",
                    "query_name": "hit_id",
                }
            )
            .drop("query_accession")
        )
    df = df.with_columns(pl.col("hit_id").cast(pl.Utf8))
    df = df.join(vibrant_names, on="hit_id", how="left")

    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
        pl.lit(db).alias("database"),
    ])

    df = df.join(
        reformat_for_merge(gene_id_mapping).lazy(),
        on=["gene", "sample"],
        how="left"
    )

    # Add sample prefixes to gene names since some scaffold names are identical across samples
    df = df.with_columns(
        pl.concat_str([pl.col("sample"), pl.lit("__"), pl.col("gene")]).alias("gene"),
    )

    df = df.with_columns(
        pl.col("gene").cast(pl.Utf8),
        pl.col("hit_id").cast(pl.Utf8),
        pl.col("database").cast(pl.Utf8),
        pl.col("evalue").cast(pl.Float64),
        pl.col("bitscore").cast(pl.Float64),
    )

    dfs_vibrant_all_annots.append(df)

In [ ]:
vibrant_all_annots = pl.concat(dfs_vibrant_all_annots, how="diagonal_relaxed")

## Combine LazyFrames of all annotations from each tool and filter to just AMG genes

In [ ]:
AMG_PREDICTIONS_COMBINED = AMG_TABLES_DIR.joinpath("amg_predictions_combined.parquet")

In [ ]:
amg_genes = pl.read_parquet(AMG_PREDICTIONS_COMBINED)
assert_benchmark_samples(amg_genes, BENCHMARK_SAMPLES, "amg_predictions_combined.parquet")

In [ ]:
AMG_GENE_LIST = amg_genes.get_column("gene").to_list()

In [ ]:
def _normalize_pfam_id(expr: pl.Expr) -> pl.Expr:
    return (
        pl.when(expr.is_not_null() & expr.str.starts_with("PF"))
        .then(expr.str.replace(r"\.\d+$", ""))
        .otherwise(expr)
    )

### Collect the LazyFrame

**This used up to 1500 GB of RAM**

In [ ]:
checkamg_block = (
    checkamg_all_annots
    .with_columns([
        pl.lit("CheckAMG").alias("tool"),
        pl.col("hmm_id").alias("hit_id"),
        pl.col("name").alias("hit_desc"),
        pl.col("evalue").cast(pl.Float64).alias("evalue"),
        pl.col("score").cast(pl.Float64).alias("bitscore"),
    ])
    .select(["gene","source","sample","ecosystem","tool","database","hit_id","hit_desc","bitscore","evalue"])
)

dramv_block = (
    dramv_all_annots
    .with_columns([
        pl.lit("DRAMV").alias("tool"),
    ])
    .select(["gene","source","sample","ecosystem","tool","database","hit_id","hit_desc","bitscore","evalue"])
)

vibrant_block = (
    vibrant_all_annots
    .with_columns([
        pl.lit("VIBRANT").alias("tool"),
    ])
    .select(["gene","source","sample","ecosystem","tool","database","hit_id","hit_desc","bitscore","evalue"])
)

In [ ]:
def normalize_block(df, tool):
    return (
        df
        .with_columns([
            pl.lit(tool).alias("tool"),
            pl.col("gene").cast(pl.Utf8),
            pl.col("source").cast(pl.Utf8),
            pl.col("sample").cast(pl.Utf8),
            pl.col("ecosystem").cast(pl.Utf8),
            pl.col("database").cast(pl.Utf8),
            pl.col("hit_id").cast(pl.Utf8),
            pl.col("hit_desc").cast(pl.Utf8),
            pl.col("bitscore").cast(pl.Float64),
            pl.col("evalue").cast(pl.Float64)
        ])
        .select([
            "gene","source","sample","ecosystem","tool",
            "database","hit_id","hit_desc","bitscore","evalue"
        ])
    )

Display labels for ecosystem, sample type and tool. The sample type of six soil samples comes from the `sample_type_corrections` params, keyed by sample name, because their directory placement is reversed. Every `source` value in this notebook passes through this function, so the annotation tables and the raw final-result tables join on the same gene key.

In [ ]:
from sample_types import apply_sample_type_corrections, assert_sample_types, sample_type_corrections

SAMPLE_TYPE_CORRECTIONS = sample_type_corrections()


def reformat_source_ecosystem_tool(df):
    df = df.with_columns([
        pl.col("ecosystem")
        .str.replace_all("gut", "Human gut", literal=True)
        .str.replace_all("freshwater", "Aquatic", literal=True)
        .str.replace_all("marine", "Aquatic", literal=True)
        .str.replace_all("soil", "Soil", literal=True)
        .str.replace_all("aquatic", "Aquatic", literal=True)
        .alias("ecosystem"),

        pl.col("source")
        .str.replace_all("metagenomes", "Mixed metagenomes", literal=True)
        .str.replace_all("viromes", "Viromes", literal=True)
        .str.replace_all("complete_virus_genomes", "Viral genomes", literal=True)
        .alias("source"),
    ])

    if "tool" in df.columns:
        df = df.with_columns([
            pl.col("tool")
            .str.replace_all("checkamg", "CheckAMG", literal=True)
            .str.replace_all("vibrant", "VIBRANT", literal=True)
            .str.replace_all("dramv", "DRAM-V", literal=True)
            .str.replace_all("DRAMV", "DRAM-V", literal=True)
            .alias("tool"),
        ])

    return apply_sample_type_corrections(df, SAMPLE_TYPE_CORRECTIONS)

In [ ]:
amg_annots = (
    pl.concat([
        normalize_block(checkamg_block.lazy(),"CheckAMG"),
        normalize_block(dramv_block.lazy(),"DRAMV"),
        normalize_block(vibrant_block.lazy(),"VIBRANT")
    ])
    # Change gene names back and remove version suffices from Pfam IDs
    .with_columns([
        pl.col("gene").str.split("__").list.get(-1).alias("gene"),
        _normalize_pfam_id(pl.col("hit_id").cast(pl.Utf8)).alias("hit_id")

    ])
    # Filter to just the genes that were predicted as AMGs in the same sample
    .join(amg_genes.lazy().select("sample", "gene").unique(), on=["sample", "gene"], how="semi")
    # Keep hits with evalue <= 1e-3 and bitscore >= 30, which still includes many non-final hits
    .filter(
        (pl.col("evalue") <= 1e-3) & (pl.col("bitscore") >= 30)
    )
)

In [ ]:
amg_annots = reformat_source_ecosystem_tool(amg_annots)

In [ ]:
checkamg_all_annots, dramv_all_annots, vibrant_all_annots = None, None, None
checkamg_block, dramv_block, vibrant_block = None, None, None

## Label annotations that made it into the final annotations / filtered hits for each tool

### Load the original, combined final results from each tool

Need to load the combined "raw" results from each tool and join by gene, annotation, and database to infer whether an anntoation made it into the final hits for a tool.

#### CheckAMG

In [ ]:
checkamg_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("checkamg_results_raw.parquet"))

In [ ]:
base_cols = ["source", "sample", "ecosystem", "gene"]

dfs_checkamg_raw = []

# KEGG
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["KEGG KO", "KEGG KO Name"])
    .rename({"KEGG KO": "id", "KEGG KO Name": "name"})
    .with_columns(pl.lit("KEGG").alias("db"))
)

# FOAM
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["FOAM ID", "FOAM Annotation"])
    .rename({"FOAM ID": "id", "FOAM Annotation": "name"})
    .with_columns(pl.lit("FOAM").alias("db"))
)

# Pfam
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["Pfam Accession", "Pfam Name"])
    .rename({"Pfam Accession": "id", "Pfam Name": "name"})
    .with_columns(pl.lit("Pfam").alias("db"))
)

# CAZy
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["CAZy Family", "CAZy Activities"])
    .rename({"CAZy Family": "id", "CAZy Activities": "name"})
    .with_columns(pl.lit("CAZy").alias("db"))
)

# METABOLIC
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["METABOLIC db ID", "METABOLIC Annotation"])
    .rename({"METABOLIC db ID": "id", "METABOLIC Annotation": "name"})
    .with_columns(pl.lit("METABOLIC").alias("db"))
)

# CAMPER
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["CAMPER ID", "CAMPER Annotation"])
    .rename({"CAMPER ID": "id", "CAMPER Annotation": "name"})
    .with_columns(pl.lit("CAMPER").alias("db"))
)

# PHROG
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["PHROG Number", "PHROG Annotation"])
    .rename({"PHROG Number": "id", "PHROG Annotation": "name"})
    .with_columns(pl.lit("PHROG").alias("db"))
)

# combine + drop null ids
checkamg_results_raw_db_long = (
    pl.concat(dfs_checkamg_raw, how="vertical_relaxed")
    .filter(pl.col("id").is_not_null())
    .with_columns([
        _normalize_pfam_id(pl.col("id").cast(pl.Utf8)).alias("id"),
        pl.lit("CheckAMG").alias("tool")
        ])
)

checkamg_results_raw_db_long = reformat_source_ecosystem_tool(checkamg_results_raw_db_long)

In [ ]:
checkamg_results_raw_db_long

#### DRAM-V

In [ ]:
dramv_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("dramv_results_raw.parquet"))

In [ ]:
dramv_results_raw = (
    dramv_results_raw
    .select([
        "source", "sample", "ecosystem", "gene",
        "gene_id", "gene_description",
    ])
    .rename({"gene_id": "id", "gene_description": "name"})
    .with_columns([
        _normalize_pfam_id(pl.col("id").cast(pl.Utf8)).alias("id"),
        pl.lit("DRAM-V").alias("tool")
    ])
)
dramv_results_raw = reformat_source_ecosystem_tool(dramv_results_raw)
dramv_results_raw

#### VIBRANT

In [ ]:
vibrant_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("vibrant_results_raw.parquet"))

In [ ]:
dfs_vibrant_raw = []

# KEGG (AMG KO)
dfs_vibrant_raw.append(
    vibrant_results_raw
    .select(base_cols + ["AMG KO", "AMG KO name"])
    .rename({"AMG KO": "id", "AMG KO name": "name"})
    .with_columns(pl.lit("KEGG").alias("db"))
)

# Pfam
dfs_vibrant_raw.append(
    vibrant_results_raw
    .select(base_cols + ["Pfam", "Pfam name"])
    .rename({"Pfam": "id", "Pfam name": "name"})
    .with_columns(pl.lit("Pfam").alias("db"))
)

vibrant_results_raw_db_long = (
    pl.concat(dfs_vibrant_raw, how="vertical_relaxed")
    .filter(pl.col("id").is_not_null() & (pl.col("id") != ""))
    .with_columns([
        _normalize_pfam_id(pl.col("id").cast(pl.Utf8)).alias("id"),
        pl.lit("VIBRANT").alias("tool")
    ])
)

vibrant_results_raw_db_long = reformat_source_ecosystem_tool(vibrant_results_raw_db_long)

In [ ]:
vibrant_results_raw_db_long

The final-result tables and the combined AMG predictions must carry the corrected sample type for the six soil samples, or the final-annotation joins below on (source, sample, ecosystem, gene) would miss their rows.

In [ ]:
for name, frame in (("CheckAMG final results", checkamg_results_raw_db_long), ("DRAM-V final results", dramv_results_raw),
                    ("VIBRANT final results", vibrant_results_raw_db_long), ("amg_predictions_combined.parquet", amg_genes)):
    assert_sample_types(frame, SAMPLE_TYPE_CORRECTIONS, name)

### Join with AMG anntoations and label the filtered/final annots

In [ ]:
amg_annots_checkamg_labeled = (
    amg_annots
    .filter(pl.col("tool") == "CheckAMG")
    .join(
        (
            checkamg_results_raw_db_long
            .select(["source", "sample", "ecosystem", "gene", "db", "id"])
            .with_columns(pl.lit(True).alias("final_annot"))
            .lazy()
        ),
        left_on=["source", "sample", "ecosystem", "gene", "database", "hit_id"],
        right_on=["source", "sample", "ecosystem", "gene", "db", "id"],
        how="left"
    )
    .unique()
    .sort(["source", "sample", "ecosystem", "gene", "evalue"], descending=[False, False, False, False, False])
    .with_columns(pl.col("final_annot").fill_null(False))
)

In [ ]:
amg_annots_dramv_labeled = (
    amg_annots
    .filter(pl.col("tool") == "DRAM-V")
    .join(
        (
            dramv_results_raw
            .select(["source", "sample", "ecosystem", "gene", "id"])
            .with_columns(pl.lit(True).alias("final_annot"))
            .lazy()
        ),
        left_on=["source", "sample", "ecosystem", "gene", "hit_id"],
        right_on=["source", "sample", "ecosystem", "gene", "id"],
        how="left"
    )
    .unique()
    .sort(["source", "sample", "ecosystem", "gene", "evalue"], descending=[False, False, False, False, False])
    .with_columns(pl.col("final_annot").fill_null(False))
)

In [ ]:
amg_annots_vibrant_labeled = (
    amg_annots
    .filter(pl.col("tool") == "VIBRANT")
    .join(
        (
            vibrant_results_raw_db_long
            .select(["source", "sample", "ecosystem", "gene", "id", "db"])
            .with_columns(pl.lit(True).alias("final_annot"))
            .lazy()
        ),
        left_on=["source", "sample", "ecosystem", "gene", "hit_id", "database"],
        right_on=["source", "sample", "ecosystem", "gene", "id", "db"],
        how="left"
    )
    .unique()
    .sort(["source", "sample", "ecosystem", "gene", "evalue"], descending=[False, False, False, False, False])
    .with_columns(pl.col("final_annot").fill_null(False))
)

The `.collect()` operation will use **up to ~1500 GB of RAM**. To avoid this, `.collect()` can be removed and the annotation table can be written directly using `.sink_parquet()`.

In [ ]:
amg_annots_labeled = pl.concat([
    amg_annots_checkamg_labeled,
    amg_annots_dramv_labeled,
    amg_annots_vibrant_labeled
])

In [ ]:
amg_annots_labeled.collect_schema()

In [ ]:
AMG_ANNOTATIONS_OUTPUT = AMG_PREDICTIONS_COMBINED.parent.joinpath("amg_all_annotations.parquet")

In [ ]:
amg_annots_labeled.sink_parquet(AMG_ANNOTATIONS_OUTPUT)
n_rows = pl.scan_parquet(AMG_ANNOTATIONS_OUTPUT).select(pl.len()).collect().item()
print(f"{AMG_ANNOTATIONS_OUTPUT}: {n_rows:,} rows")
assert_sample_types(pl.scan_parquet(AMG_ANNOTATIONS_OUTPUT), SAMPLE_TYPE_CORRECTIONS, "amg_all_annotations.parquet")